In [ ]:
import uproot
import awkward as ak

import pandas as pd

import matplotlib.pylab as plt
import numpy as np

import hist
from hist import Hist

import time

import myPIDselector

import math

import os

data = ak.from_parquet("Background_and_signal_SP_modes_All_runs.parquet")
blindD = ak.from_parquet("Data_All_runs_BLINDED.parquet")

B_D1_IDX_FIELD            = 'Bd1Idx'
B_D2_IDX_FIELD            = 'Bd2Idx'

# Lambda daughter fields
LAMBDA_PROTON_IDX_FIELD   = 'Lambda0d1Idx'
LAMBDA_PROTON_LUND_FIELD  = 'Lambda0d1Lund'

 
PROTON_TRKIDX_FIELD       = 'pTrkIdx'
PROTON_SELECTORS_FIELD    = 'pSelectorsMap'
TRK_LUND_FIELD            = 'TRKLund'

MES_FIELD                 = 'BpostFitMes'
DELTAE_FIELD              = 'BpostFitDeltaE'

SPMODE_FIELD              = 'spmode'

plt.close()

def calculate_bits_for_PID_selector(trkidx, trk_selector_map, verbose=0):
    
    bits = None

    # If there is no trk index passed in, just calculate the bits for
    # all of the tracks
    if trkidx is None:
        trkidx = ak.local_index(trk_selector_map)

    # Grab the tracks that map on to the particle/collection we are interested in 
    subset_of_trk_selector_map = trk_selector_map[trkidx]
    if verbose:
        print("values in the subset of the trk selector map")
        print(subset_of_trk_selector_map)
        print()
        
    # We need this to convert the numbers in the selector to binary
    binary_repr_vec = np.vectorize(np.binary_repr)

    # Grab the number of entries in each so we can unflatten this later
    counts = ak.num(subset_of_trk_selector_map)
    
    # Now get the binary representation (as a string) for the flattened subset
    binrep = binary_repr_vec(ak.flatten(subset_of_trk_selector_map), width=32)

    if verbose:
        print("binary representation of selector map")
        print(binrep)
        print()

    # Convert the string to integers
    tempbits = np.array(binrep).astype(int)
    bits = ak.unflatten(tempbits,counts)

    if verbose:
        print("flattened integer representation of selectors as binary (int)")
        print(tempbits)
        print()
        print("unflattened integer representation of selectors as binary (int)")
        print(bits)
        print()

    return bits
def mask_PID_selection(bits, selector, pid_map_object):


    bit_to_look_for = pid_map_object.selectors.index(selector)

    place = int(math.pow(10,bit_to_look_for))
    #print(place)

    mask = bits // place % 10

    mask_bool = ak.values_astype(mask,bool)

    return mask_bool
def get_info_for_PID_masks(ak_arr, verbosity=0):

    # Proton and pion information from the Lambda decay
    # These are the index of the proton (d1) and pion (d2) in those lists
    L1d1idx = ak_arr.Lambda0d1Idx[ak_arr.Bd1Idx]
    L1d2idx = ak_arr.Lambda0d2Idx[ak_arr.Bd1Idx]

    L2d1idx = ak_arr.Lambda0d1Idx[ak_arr.Bd2Idx]
    L2d2idx = ak_arr.Lambda0d2Idx[ak_arr.Bd2Idx]


    
     
    #d1lund = ak_arr['Lambda0d1Lund']
    #d2lund = ak_arr['Lambda0d2Lund']
    
    #Bd2idx = ak_arr['Bd2Idx']
    #Bd2lund = ak_arr['Bd2Lund']
    
    trkidx_proton = ak_arr['pTrkIdx']
    trk_selector_map_proton = ak_arr['pSelectorsMap']
    
    trkidx_pion = ak_arr['piTrkIdx']
    trk_selector_map_pion = ak_arr['piSelectorsMap']

    indices_and_maps = {}
    indices_and_maps['L1d1idx'] = L1d1idx
    indices_and_maps['L1d2idx'] = L1d2idx
    indices_and_maps['L2d1idx'] = L2d1idx
    indices_and_maps['L2d2idx'] = L2d2idx
    #indices_and_maps['Bd2idx'] = Bd2idx

    indices_and_maps['trkidx_proton'] = trkidx_proton
    indices_and_maps['trk_selector_map_proton'] = trk_selector_map_proton

    indices_and_maps['trkidx_pion'] = trkidx_pion
    indices_and_maps['trk_selector_map_pion'] = trk_selector_map_pion

    return indices_and_maps
def PID_masks(indices_and_maps, \
              lam1p_selector,\
              lam1pi_selector, \
              lam2p_selector,\
              lam2pi_selector, \
             verbosity=0):
                #change Bpselector
    L1d1idx = indices_and_maps['L1d1idx']
    L1d2idx = indices_and_maps['L1d2idx']
    L2d1idx = indices_and_maps['L2d1idx']
    L2d2idx = indices_and_maps['L2d2idx']
    #Bd2idx = indices_and_maps['Bd2idx']

    trkidx_proton = indices_and_maps['trkidx_proton']
    trk_selector_map_proton = indices_and_maps['trk_selector_map_proton']

    trkidx_pion = indices_and_maps['trkidx_pion']
    trk_selector_map_pion = indices_and_maps['trk_selector_map_pion']
    
    # Proton
    pbits = calculate_bits_for_PID_selector(trkidx_proton, trk_selector_map_proton, verbose=verbosity)
    # Pion
    pibits = calculate_bits_for_PID_selector(trkidx_pion, trk_selector_map_pion, verbose=verbosity)
    
    
    mask_bool_protonL1 = mask_PID_selection(pbits[L1d1idx], lam1p_selector, pps)
        
    mask_bool_pionL1 = mask_PID_selection(pibits[L1d2idx], lam1pi_selector, pips)

    mask_bool_protonL2 = mask_PID_selection(pbits[L2d1idx], lam2p_selector, pps)
        
    mask_bool_pionL2 = mask_PID_selection(pibits[L2d2idx], lam2pi_selector, pips)

    return mask_bool_protonL1, mask_bool_pionL1,mask_bool_protonL2, mask_bool_pionL2
def indices_to_booleans(indices, array_to_slice):
    """Convert ragged integer indices to a boolean mask over array_to_slice."""
    whole_set, in_set = ak.unzip(ak.cartesian(
        [ak.local_index(array_to_slice), indices], nested=True
    ))
    return ak.any(whole_set == in_set, axis=-1)
def ragged_element_pick(ragged_arr, flat_idx):
    """
    For each event i, return ragged_arr[i][flat_idx[i]].

    Parameters
    ----------
    ragged_arr : ak.Array, shape [n_events, var]
    flat_idx   : array-like, shape [n_events]  (integer index per event)

    Returns
    -------
    np.ndarray, shape [n_events]
    """
    flat_content = ak.flatten(ragged_arr)
    counts       = ak.num(ragged_arr)

    offsets = np.zeros(len(counts) + 1, dtype=np.int64)
    np.cumsum(ak.to_numpy(counts), out=offsets[1:])

    idx_np = ak.to_numpy(flat_idx).astype(np.int64)
    global_indices = offsets[:-1] + idx_np

    return ak.to_numpy(flat_content)[global_indices]
def build_antilambda_antimask(ak_arr, pps,
                               selector,
                               verbose=0):
    """
    Antibaryon veto for B0 -> Lambda Lambda (BNV).

    REQUIRES: single-B-candidate cut already applied (exactly one B per event).

    Returns
    -------
    charge_test : awkward array of booleans
        Per-track flag — True where a rest-of-event track is an antiproton.
    """

    index = 721

    n = len(ak_arr)

     # 1.  Squeeze B-candidate dimension
    #bd1idx = ak.to_numpy(ak_arr[B_D1_IDX_FIELD][:, 0])   # scalar per event
    bd1idx = ak.to_numpy(ak.firsts(ak_arr[B_D1_IDX_FIELD], axis=1))
    

    #bd2idx = ak.to_numpy(ak_arr[B_D2_IDX_FIELD][:, 0])
    bd2idx = ak.to_numpy(ak.firsts(ak_arr[B_D2_IDX_FIELD], axis=1))
    

    if verbose:
        print(f"bd1idx  {bd1idx[:5]}")
        print(f"bd2idx  {bd2idx[:5]}")
        print()

     
    # 2.  Chain: B daughter Lambda index -> that Lambda's proton index
     
    all_lam_proton_idx  = ak_arr[LAMBDA_PROTON_IDX_FIELD]
    all_lam_proton_lund = ak_arr[LAMBDA_PROTON_LUND_FIELD]

    # Proton-list index for each B-daughter Lambda (flat numpy, one per event)
    lam1_pidx = ragged_element_pick(all_lam_proton_idx, bd1idx)
    lam2_pidx = ragged_element_pick(all_lam_proton_idx, bd2idx)
    

    if verbose:
        print(f"lam1_pidx (proton-list index from Λ₁)  {lam1_pidx[:5]}")
        print(f"lam2_pidx (proton-list index from Λ₂)  {lam2_pidx[:5]}")
        print()

     #     qlam1 = +1 for Λ → p π⁻,  -1 for Λ̄ → p̄ π⁺
    lam1_lund = ragged_element_pick(all_lam_proton_lund, bd1idx)
    lam2_lund = ragged_element_pick(all_lam_proton_lund, bd2idx)
    

    qlam1 = np.sign(lam1_lund).astype(float)   # shapes
    qlam2 = np.sign(lam2_lund).astype(float)
    

    if verbose:
        print(f"qlam1 (charge of Λ₁ proton)  {qlam1[:5]}")
        print(f"qlam2 (charge of Λ₂ proton)  {qlam2[:5]}")
        print()

    qtrk = ak_arr[TRK_LUND_FIELD] / np.abs(ak_arr[TRK_LUND_FIELD])
    

     # 4.  Map proton-list indices to track-list indices
    trk_selector_map = ak_arr[PROTON_SELECTORS_FIELD]
    trkidx_proton    = ak_arr[PROTON_TRKIDX_FIELD]

    # Pick the track index for each signal proton (flat numpy)
    lam1_trkidx_flat = ragged_element_pick(trkidx_proton, lam1_pidx)
    lam2_trkidx_flat = ragged_element_pick(trkidx_proton, lam2_pidx)

    # Wrap as length-1 ragged for indices_to_booleans
    lam1_trkidx = ak.unflatten(lam1_trkidx_flat, 1)
    lam2_trkidx = ak.unflatten(lam2_trkidx_flat, 1)
    if verbose:
        print(f"lam1_trkidx  {lam1_trkidx[:5]}")
        print(f"lam2_trkidx  {lam2_trkidx[:5]}")
        print()

    bool_idx1 = indices_to_booleans(lam1_trkidx, trk_selector_map)
    bool_idx2 = indices_to_booleans(lam2_trkidx, trk_selector_map)


    if verbose:
        print(f"bool_idx1 (Λ₁ proton)  {bool_idx1[:3]}")
        print(f"bool_idx2 (Λ₂ proton)  {bool_idx2[:3]}")
        print()

    
    # 5.  PID selection on ALL proton candidates  
    pbits     = calculate_bits_for_PID_selector(None, trk_selector_map)
    mask_bool = mask_PID_selection(pbits, selector, pps)

    if verbose:
        print(f"Tracks passing {selector}:  {ak.sum(ak.num(pbits[mask_bool]))}")
        print()

     # 6.  Charge test on REST-OF-EVENT proton candidates
     
    other_mask  = ~(bool_idx1 | bool_idx2) & mask_bool
    charge_test = qtrk[other_mask] == -qlam1


    if verbose:
        print(f"charge_test  {charge_test[:3]}")

   
    have_opp       = ak.num(charge_test[charge_test])
    nhave_opp      = len(have_opp[have_opp > 0])
    ndont_have_opp = len(have_opp[have_opp == 0])

    return charge_test
def apply_antibaryon_veto(ak_arr, pps, selector = 'LooseKMProtonSelection'):
    """Return a per-event boolean: True = passes veto (no antiproton found)."""
    ct       = build_antilambda_antimask(ak_arr, pps, selector=selector, verbose=0)
    have_opp = ak.num(ct[ct])
    return have_opp == 0

pps = myPIDselector.PIDselector("p")
pips = myPIDselector.PIDselector("pi")

pidChoiceDict = {}
DD = {0: 963, 1: 963, 256: 291, 112: 731, 157: 745, -1: 226}
D998 = {0: 231.27929028504946, 1: 231.27929028504946, 256: 103.9152414194299, 112: 183.88801628853986, 157: 188.33094822571263, -1: 86.1435136707388}
D1005 = {0: 293.19359668317406, 1: 293.19359668317406, 256: 39.581135552228496, 112: 207.1901416561096, 157: 209.14476563399748, -1: 21.500863756766098}
Dbkg={}
minSigCut = 10
maxSigCut = 10
for pidChoice in [0,1,256,112,157,-1]:
    Dbkg[pidChoice] = D998[pidChoice]+D1005[pidChoice]
for cutI in [0]:
    for pidChoice in [0]:# 157, 202]:#,1 112, 256, 121]:
        if pidChoice == 1:
            myPMask = "SuperLooseKMProtonSelection"
            myPShort = "SL Pro"
            myPiMask = "SuperLooseKMPionMicroSelection"
            myPiShort = 'SL Pi'
            pidChoiceDict[pidChoice] = (myPShort,myPiShort)
        elif pidChoice == 157:
            myPMask = "VeryLooseKMProtonSelection"
            myPShort = "VL Pro"
            myPiMask = "VeryLooseKMPionMicroSelection"
            myPiShort = 'VL Pi'
            pidChoiceDict[pidChoice] = (myPShort,myPiShort)
        elif pidChoice == 202:
            myPMask = "VeryLooseKMProtonSelection"
            myPShort = "VL Pro"
            myPiMask = "TightKMPionMicroSelection"
            myPiShort = 'T Pi'
            pidChoiceDict[pidChoice] =(myPShort,myPiShort)
        elif pidChoice == 112:
            myPMask = "VeryLooseKMProtonSelection"
            myPShort = "VL Pro"
            myPiMask = "LooseKMPionMicroSelection"
            myPiShort = 'L Pi'
            pidChoiceDict[pidChoice] = (myPShort,myPiShort)
        elif pidChoice == 256:
            myPMask = "LooseKMProtonSelection"
            myPShort = "L Pro"
            myPiMask = "VeryLooseKMPionMicroSelection"
            myPiShort = 'VL Pi'
            pidChoiceDict[pidChoice] = (myPShort,myPiShort)
        elif pidChoice == 121:
            myPMask = "SuperLooseKMProtonSelection"
            myPShort = "SL Pro"
            myPiMask = "SuperTightKMPionMicroSelection"
            myPiShort = 'ST Pi'
            pidChoiceDict[pidChoice] = (myPShort,myPiShort)
        elif pidChoice == -1:
            myPMask = "TightKMProtonSelection"
            myPShort = "T Pro"
            myPiMask = "VeryLooseKMPionMicroSelection"
            myPiShort = 'VL Pi'
            pidChoiceDict[pidChoice] = (myPShort,myPiShort)
        elif pidChoice == -2:
            myPMask = "VeryTightKMProtonSelection"
            myPShort = "VT Pro"
            myPiMask = "VeryLooseKMPionMicroSelection"
            myPiShort = 'VL Pi'
            pidChoiceDict[pidChoice] = (myPShort,myPiShort)
        elif pidChoice == -3:
            myPMask = "SuperTightKMProtonSelection"
            myPShort = "ST Pro"
            myPiMask = "VeryLooseKMPionMicroSelection"
            myPiShort = 'VL Pi'
            pidChoiceDict[pidChoice] = (myPShort,myPiShort)

        elif pidChoice == 0:
            myPMask =  "SuperLooseKMProtonSelection"
            myPiMask = "SuperLooseKMPionMicroSelection"
            pidChoiceDict[pidChoice] = ("None","None")

        
        

        d1b = data[(data['nB']==1)]

        indices_and_maps = get_info_for_PID_masks(d1b)

        mask_bool_protonL1, mask_bool_pionL1,mask_bool_protonL2, mask_bool_pionL2 = PID_masks(indices_and_maps, \
                    lam1p_selector=myPMask, \
                    lam1pi_selector=myPiMask, \
                    lam2p_selector=myPMask, \
                    lam2pi_selector=myPiMask, \
                    verbosity=0)
        
        mes = d1b['BpostFitMes']
        de = d1b['BpostFitDeltaE']

        meslo = 5.2
        mesloS = 5.27
        meshiS = 5.3
        DeltaElo = -0.2
        DeltaEhi = 0.2
        DeltaEloS = -0.07
        DeltaEhiS = 0.07

        mask_pid = mask_bool_pionL1 & mask_bool_protonL1 & mask_bool_pionL2 & mask_bool_protonL2

        preSel_fit = (mes>meslo)&(de<DeltaEhi)&(de>DeltaElo)

        mask_sig = (mes>=mesloS) & (mes<=meshiS) & (de>=DeltaEloS) & (de<=DeltaEhiS)

        mask_sideband = (((mes<mesloS) | (mes>meshiS)) | ((de<DeltaEloS) | (de>DeltaEhiS)))

        firstMask = mask_pid & preSel_fit
        
        
        

        lamconflsig = d1b['Lambda0postFitFlightSignificance']
        lamuncmass = d1b['Lambda0_unc_Mass']
        BCTT = d1b['BCosThetaT']
        BCTS = d1b['BCosThetaS']
        bctsMask = (BCTS<0.95)

        lammass_world_average = 1.115683
        width = 0.003
        lo = lammass_world_average - width
        hi = lammass_world_average + width
            

        mask_fl = (lamconflsig>=minSigCut)
        mask_upper_fl = (lamconflsig>=maxSigCut)
        flightSigMask = (ak.all(mask_fl,axis=1))&(ak.any(mask_upper_fl, axis=1))

        lamuncmassMask = (lamuncmass[:,0]>=lo)&(lamuncmass[:,0]<=hi)& (lamuncmass[:,1]>=lo)&(lamuncmass[:,1]<=hi)
        vetoMask = apply_antibaryon_veto(d1b, pps)

        finalMask = flightSigMask&lamuncmassMask&preSel_fit&mask_pid&vetoMask

        in_signal = preSel_fit&mask_sig
        in_sideband = preSel_fit&mask_sideband

        scaleD = {"998": 424.3/1719, "1005":424.3/868.3, "-999":1,'1235':424.3/1289,'1237':424.3/1306}

        
        binCount = 30
        
        bins = np.linspace(-1, 1, binCount)
        fig, ax = plt.subplots(figsize=(10, 6))

        stackMult999 = 963/1312
        stackMult999bcts = 586/1198
        stackMult999=43/478

        stackMult = 963/525
        stackMultbcts = 586/348
        stackMult = 43/20


        for sp in ['998','1005','1235','1237','-999']:
            
            spmask = (d1b['spmode']==sp)

            bcttVals = BCTT[firstMask&spmask&in_sideband&bctsMask&finalMask]
            x = ak.to_numpy(ak.flatten(bcttVals))

            if sp =='998':
                x998 = x
                weights998 = np.ones_like(x998) * scaleD[sp] * stackMult
                counts998,bins998 = np.histogram(x998, bins=bins, range=(-1,1))
                #print(sum(counts998))
                
            elif sp =='1005':
                x1005 = x
                weights1005 = np.ones_like(x1005) * scaleD[sp] * stackMult
                counts1005,bins1005 = np.histogram(x1005, bins=bins, range=(-1,1))
                #print(sum(counts1005))

            elif sp =='1235':
                x1235 = x
                weights1235 = np.ones_like(x1235) * scaleD[sp] * stackMult
                counts1235,bins1235 = np.histogram(x1235, bins=bins, range=(-1,1))
                #print(sum(counts1235))

            elif sp =='1237':
                x1237 = x
                weights1237 = np.ones_like(x1237) * scaleD[sp] * stackMult
                counts1237,bins1237 = np.histogram(x1237, bins=bins, range=(-1,1))
                #print(sum(counts1237))

            elif sp == '-999':
                x999 = x
                weights999 = np.ones_like(x999) * stackMult999
                counts999,bins999 = np.histogram(x999, bins=bins, range=(-1,1))
                print("999", sum(counts999))


        print("SUM BKG",sum(counts1005)*(424.3/868.3)+sum(counts998)*(424.3/1719)+sum(counts1235)*(424.3/1289)+sum(counts1237)*(424.3/1306))
            
        ax.hist([x998, x1005], bins=bins, weights= [weights998,weights1005], stacked=True, label=['998', '1005'], alpha=0.7, color=['#1f77b4', '#aec7e8'])

        ax.hist(x999, bins=bins, weights = [weights999], histtype='step', linestyle='--', linewidth=2, color='#ff7f0e', label='-999')
                

        d1b = blindD[(blindD['nB']==1)]

        indices_and_maps = get_info_for_PID_masks(d1b)

        mask_bool_protonL1, mask_bool_pionL1,mask_bool_protonL2, mask_bool_pionL2 = PID_masks(indices_and_maps, \
                    lam1p_selector=myPMask, \
                    lam1pi_selector=myPiMask, \
                    lam2p_selector=myPMask, \
                    lam2pi_selector=myPiMask, \
                    verbosity=0)
        
        mes = d1b['BpostFitMes']
        de = d1b['BpostFitDeltaE']

        meslo = 5.2
        mesloS = 5.27
        meshiS = 5.3
        DeltaElo = -0.2
        DeltaEhi = 0.2
        DeltaEloS = -0.07
        DeltaEhiS = 0.07

        mask_pid = mask_bool_pionL1 & mask_bool_protonL1 & mask_bool_pionL2 & mask_bool_protonL2

        preSel_fit = (mes>meslo)&(de<DeltaEhi)&(de>DeltaElo)

        mask_sig = (mes>=mesloS) & (mes<=meshiS) & (de>=DeltaEloS) & (de<=DeltaEhiS)

        mask_sideband = (((mes<mesloS) | (mes>meshiS)) | ((de<DeltaEloS) | (de>DeltaEhiS)))
        
        
        

        lamconflsig = d1b['Lambda0postFitFlightSignificance']
        lamuncmass = d1b['Lambda0_unc_Mass']
        BCTT = d1b['BCosThetaT']
        BCTS = d1b['BCosThetaS']
        bctsMask = (BCTS<0.95)

        lammass_world_average = 1.115683
        width = 0.003
        lo = lammass_world_average - width
        hi = lammass_world_average + width
            

        mask_fl = (lamconflsig>=minSigCut)
        mask_upper_fl = (lamconflsig>=maxSigCut)
        flightSigMask = (ak.all(mask_fl,axis=1))&(ak.any(mask_upper_fl, axis=1))

        lamuncmassMask = (lamuncmass[:,0]>=lo)&(lamuncmass[:,0]<=hi)& (lamuncmass[:,1]>=lo)&(lamuncmass[:,1]<=hi)
        vetoMask = apply_antibaryon_veto(d1b, pps)

        finalMask = flightSigMask&lamuncmassMask&preSel_fit&mask_pid&vetoMask

        in_signal = preSel_fit&mask_sig
        in_sideband = preSel_fit&mask_sideband

        bcttVals = BCTT[preSel_fit&mask_pid&bctsMask&finalMask]

        x = ak.to_numpy(ak.flatten(bcttVals))
        counts, bin_edges = np.histogram(x, bins=bins, range = (-1,1))

        print("DATA", sum(counts))

        # 3. Find the center of each bin
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

        # 4. Calculate Poisson errors (standard statistical error for counts)
        errors = np.sqrt(counts)

        # 5. Overlay the points with error bars on top of the bars
        plt.errorbar(
            bin_centers, 
            counts, 
            yerr=errors, 
            fmt='o', 
            color='purple', 
            ecolor='purple', 
            capsize=3, 
            label='Data'
        )

        plt.xlabel(f"BCosThetaT / Sideband / Scaled bkg & sig to data / BCTS<0.95")
        plt.ylabel('Counts')
        plt.legend()
        plt.show()















